In [28]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from datetime import datetime
from torch.utils.data import DataLoader
from pathlib import Path
import random
import numpy as np
import pandas as pd
import os
import json
import time
import copy
import matplotlib.pyplot as plt
from tqdm.auto import tqdm  # Auto-detect Jupyter / Terminal

In [29]:
# Memastikan CUDA (GPU) tersedia
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU terdeteksi: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU tidak terdeteksi, menggunakan CPU.")

GPU terdeteksi: NVIDIA GeForce RTX 4060 Laptop GPU


## Konfigurasi

In [30]:
# ==============================================================================
# 1. PENGATURAN SEED (REPRODUCIBILITY)
# Tujuannya agar jika eksperimen diulang, hasilnya akan tetap sama (konsisten)
# ==============================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Mengatur agar backend CuDNN berjalan deterministik
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [31]:
# ==============================================================================
# 1. KONFIGURASI PATH (FOLDER DIRECTORY)
# ==============================================================================
# Sesuaikan 'dataset/' dengan nama folder tempat Anda menyimpan data
CURRENT_DIR     = Path.cwd()

if CURRENT_DIR.name == "testing":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

OUTPUT_DIR       = PROJECT_DIR / "dataset" / "testing"
OUTPUT_MODEL     = PROJECT_DIR / "models" / "testing"

LOG_DIR = PROJECT_DIR / "logs" / "traning"


TRAIN_DIR       = OUTPUT_DIR / "train"
VAL_DIR         = OUTPUT_DIR / "val"
TEST_DIR        = PROJECT_DIR / "dataset" / "raw"  / "GCD" / "GCD" / "test" 
MODEL_SAVE_PATH = OUTPUT_MODEL / 'best_cloud_model.pth' # Nama file ketika model disimpan nanti

OUTPUT_MODEL.mkdir(parents=True, exist_ok=True)

In [32]:
# ==============================================================================
# 1. KONFIGURASI DATASET & DATALOADER
# ==============================================================================
NUM_CLASSES     = 7
IMAGE_SIZE      = 224 # Standar untuk ResNet, MobileNet, dll (224x224)
BATCH_SIZE      = 32  # Turunkan ke 16 atau 8 jika GPU lokal mengalami "Out of Memory"
NUM_WORKERS     = 0   # Gunakan 2 atau 4 (sesuai jumlah core CPU lokal Anda)
HASIL_TRAIN     = [] 

In [33]:
# ==============================================================================
# 2. KONFIGURASI MODEL & TRAINING (HYPERPARAMETERS)
# ==============================================================================
MODEL_NAME      = 'resnet50'  # Referensi nama model yang digunakan
WEIGHTS_MODEL   = 'ResNet50_Weights.IMAGENET1K_V2'
LEARNING_RATE   = 1e-4        # 0.0001 (LR kecil bagus untuk pre-trained model/fine-tuning)
EPOCHS          = 20          # Jumlah iterasi training
SEED_VALUE      = 42          # Angka seed standar

In [34]:
# ==============================================================================
# 5. EKSEKUSI KONFIGURASI AWAL (SETUP)
# ==============================================================================
# Terapkan Seed
set_seed(SEED_VALUE)

# Cek Ketersediaan GPU Lokal
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print Laporan Konfigurasi
print("-" * 50)
print("⚙️ KONFIGURASI SISTEM & HARDWARE")
print("-" * 50)
print(f"Perangkat yang digunakan : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"Nama GPU Lokal         : {torch.cuda.get_device_name(0)}")
    
print("\n" + "-" * 50)
print("📂 STATUS FOLDER DATASET")
print("-" * 50)
print(f"Folder Train ada? : {TRAIN_DIR.exists()} -> Path: {TRAIN_DIR}")
print(f"Folder Val ada?   : {VAL_DIR.exists()} -> Path: {VAL_DIR}")
print(f"Folder Test ada?  : {TEST_DIR.exists()} -> Path: {TEST_DIR}")
print("-" * 50)

--------------------------------------------------
⚙️ KONFIGURASI SISTEM & HARDWARE
--------------------------------------------------
Perangkat yang digunakan : cuda
Nama GPU Lokal         : NVIDIA GeForce RTX 4060 Laptop GPU

--------------------------------------------------
📂 STATUS FOLDER DATASET
--------------------------------------------------
Folder Train ada? : True -> Path: d:\n8n-logsiswaparalayang\cloud-classification\dataset\testing\train
Folder Val ada?   : True -> Path: d:\n8n-logsiswaparalayang\cloud-classification\dataset\testing\val
Folder Test ada?  : True -> Path: d:\n8n-logsiswaparalayang\cloud-classification\dataset\raw\GCD\GCD\test
--------------------------------------------------


## Transformasi, Dataset, dan DataLoader

In [35]:
# 1. Setup Transformasi Data
# Normalisasi menggunakan rata-rata dan standar deviasi dari ImageNet
data_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

In [36]:
# 2. Inisialisasi Dataset menggunakan variabel Pathlib
# PyTorch ImageFolder menerima objek Pathlib tanpa perlu diubah menjadi string
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=data_transforms)
val_dataset   = datasets.ImageFolder(root=VAL_DIR, transform=data_transforms)
test_dataset  = datasets.ImageFolder(root=TEST_DIR, transform=data_transforms)

# Mengekstrak nama kelas
class_names = train_dataset.classes

print("-" * 50)
print("📊 INFORMASI DATASET")
print("-" * 50)
print(f"Daftar Kelas ({len(class_names)}): {class_names}")
print(f"Jumlah Data Train : {len(train_dataset)}")
print(f"Jumlah Data Val   : {len(val_dataset)}")
print(f"Jumlah Data Test  : {len(test_dataset)}")
print("-" * 50)

--------------------------------------------------
📊 INFORMASI DATASET
--------------------------------------------------
Daftar Kelas (7): ['1_cumulus', '2_altocumulus', '3_cirrus', '4_clearsky', '5_stratocumulus', '6_cumulonimbus', '7_mixed']
Jumlah Data Train : 21000
Jumlah Data Val   : 1998
Jumlah Data Test  : 9000
--------------------------------------------------


In [37]:
# 3. Setup DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Catatan: pin_memory=True digunakan untuk mempercepat transfer data dari CPU RAM ke GPU VRAM.

## Setup Model, Loss Function, dan Optimizer

In [38]:
print("-" * 50)
print(f"🛠️ MEMBANGUN MODEL: {MODEL_NAME.upper()}")
print("-" * 50)

# 1. Load Pre-trained Model
# Menggunakan weights terbaru (IMAGENET1K_V2) untuk akurasi dasar yang lebih baik
weights = WEIGHTS_MODEL
model = models.resnet50(weights=weights)

# 2. Modifikasi Layer Terakhir (Classifier)
# Mengambil jumlah fitur input dari layer fully connected terakhir
num_ftrs = model.fc.in_features

# Mengganti layer terakhir agar outputnya 7 (sesuai jumlah kelas awan)
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

# 3. Pindahkan Model ke Perangkat (GPU/CPU)
model = model.to(DEVICE)

# 4. Tentukan Loss Function dan Optimizer
criterion = nn.CrossEntropyLoss()

# Hanya optimize parameter yang butuh update. Menggunakan Adam optimizer.
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Model berhasil disiapkan dan dipindahkan ke: {DEVICE}")
print(f"Total kelas pada layer output: {model.fc.out_features}")
print("-" * 50)

--------------------------------------------------
🛠️ MEMBANGUN MODEL: RESNET50
--------------------------------------------------
Model berhasil disiapkan dan dipindahkan ke: cuda
Total kelas pada layer output: 7
--------------------------------------------------


## Fungsi Training & Validation Loop

In [39]:
@torch.no_grad()
# fungsi  Validation

def eval_model(data_loader, model, criterion, DEVICE):
    model.eval()
    loss, accuracy = 0.0, 0.0
    
    n = len(data_loader)

    for i, data in enumerate(data_loader):
        x,y = data
        x,y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x)


        loss += (criterion(pred, y)/len(x)).item()
        pred_label = torch.argmax(pred, axis = 1)
        accuracy += (torch.sum(pred_label == y) / len(x)).item()

    return loss/n, accuracy/n 


# fungsi Traning
def train(train_loader, val_loader, model, optimizer, criterion, epochs, DEVICE, save_path):
    n = len(train_loader)

    history = HASIL_TRAIN 
    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train(True)
        count = 0
        avg_loss, avg_acc = 0.0, 0.0
        train_loss, train_correct, train_total = 0.0, 0, 0
        count = 0
        print(f"Epoch {epoch+1}/{epochs}:")
        start_time = datetime.now()

        # Wrap train_loader with tqdm.auto
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=True)

        for x, y in train_bar:
            x, y = x.to(DEVICE), y.to(DEVICE) # move data to gpu

            # Forward pass
            pred = model(x)                   # compute model prediction
            loss = criterion(pred,y)          # compute model loss

            # backpropogation
            optimizer.zero_grad()             # reset gradient calculations
            loss.backward()                   # compute gradients
            optimizer.step()                  # update model parameters via optimization step

            avg_loss += loss
            pred_label = torch.argmax(pred, axis=1)
            avg_acc += (torch.sum(pred_label == y) / len(x)).item()

            # Update progress bar suffix on the fly
            
            train_loss  += loss.item() * x.size(0)
            train_correct += torch.sum(pred_label == y).item()
            train_total += x.size(0)

            live_loss = train_loss  / train_total
            live_acc = (train_correct / train_total) * 100
            train_bar.set_postfix(loss=f"{live_loss:.4f}", acc=f"{live_acc:.2f}%")

        end_time = datetime.now()
        print(f"Time: {(end_time-start_time).seconds}s")
        print(f"Average train loss: {avg_loss/n}, Average train accuracy: {avg_acc/n}")
        val_loss, val_acc = eval_model(val_loader, model, criterion, DEVICE)
        print(f"Val loss: {val_loss}, Val accuracy: {val_acc}\n")

        # --- CATAT DATA EPOCH KE HISTORY ---
        epoch_metrics = {
            "epoch": epoch + 1,
            "train_loss": float(avg_loss / n),        # Dibagi n & dikonversi ke float murni
            "train_accuracy": float(avg_acc / n),    # Dibagi n & dikonversi ke float murni
            "val_loss": float(val_loss),             # Memastikan bertipe float
            "val_accuracy": float(val_acc),         # Memastikan bertipe float
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }
        history.append(epoch_metrics)

        # --- SAVE MODEL CHECKPOINT ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"--> Saved best model checkpoint to {save_path}\n")
        else:
            print("\n")

    with open(LOG_DIR / "log_train_baseline.json", "w", encoding="utf-8") as json_file:
        json.dump(history, json_file, indent=4, ensure_ascii=False)
    print("Berhasil menyimpan log ke 'log_train_baselini.json'")

    summary_train = pd.DataFrame(history)
    summary_train.to_csv(LOG_DIR / "tabel_log_train_baseline.csv",index=False)
    print("Berhasil menyimpan tabel ke 'tabel_log_train_baseline.csv'")

In [40]:
train(train_loader, val_loader, model, optimizer, criterion, EPOCHS, DEVICE,MODEL_SAVE_PATH)

Epoch 1/20:


Epoch 1/20 [Train]:   0%|          | 0/657 [00:00<?, ?it/s]

Time: 673s
Average train loss: 0.401798278093338, Average train accuracy: 0.8625380517503806
Val loss: 0.008269363303764887, Val accuracy: 0.9083049893379211



C:\Users\jardm\AppData\Local\Temp\ipykernel_13540\2130447262.py:77: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  "train_loss": float(avg_loss / n),        # Dibagi n & dikonversi ke float murni


--> Saved best model checkpoint to d:\n8n-logsiswaparalayang\cloud-classification\models\testing\best_cloud_model.pth

Epoch 2/20:


...

Time: 398s
Average train loss: 0.02574114501476288, Average train accuracy: 0.9921518264840182
Val loss: 0.01168545899175418, Val accuracy: 0.9107851480680799



Epoch 19/20:


Epoch 19/20 [Train]:   0%|          | 0/657 [00:00<?, ?it/s]

Time: 202s
Average train loss: 0.017009716480970383, Average train accuracy: 0.9948154490106544
Val loss: 0.01599281070520235, Val accuracy: 0.9100765311528766



Epoch 20/20:


Epoch 20/20 [Train]:   0%|          | 0/657 [00:00<?, ?it/s]

Time: 215s
Average train loss: 0.01649589091539383, Average train accuracy: 0.994625190258752
Val loss: 0.012505475794697189, Val accuracy: 0.917800453920213



Berhasil menyimpan log ke 'log_train_baselini.json'
Berhasil menyimpan tabel ke 'tabel_log_train_baseline.csv'
